In [ ]:
import numpy as np

from scipy.constants import e, c

from scipy.constants import physical_constants

import os

In [ ]:
from cpymad.madx import Madx

import sixtracklib as stl
import pysixtrack

# Versioning

## SixTrackLib

In [ ]:
f = os.path.dirname(stl.__file__)
f

In [ ]:
!cd $f && git log | head -4

## PySixTrack

In [ ]:
f = os.path.dirname(pysixtrack.__file__)
f

In [ ]:
!cd $f && git log | head -4

# Parameters

In [ ]:
nmass = physical_constants['atomic mass constant energy equivalent in MeV'][0] * 1e-3
# nmass = 0.931494061 # MAD-X value

In [ ]:
A = 238
Q = 28

Ekin_per_nucleon = 0.2e9 # in eV

###

mass = A * nmass * 1e9 * e / c**2 # in kg
charge = Q * e # in Coul

Ekin = Ekin_per_nucleon * A
p0c = np.sqrt(Ekin**2 + 2*Ekin*mass/e * c**2) # in eV

Etot = np.sqrt(p0c**2 + (mass/e)**2 * c**4) * 1e-9 # in GeV
p0 = p0c / c * e # in SI units
gamma = np.sqrt(1 + (p0 / (mass * c))**2)
beta = np.sqrt(1 - gamma**-2)

## MAD-X setup

In [ ]:
madx = Madx()
madx.options.echo = False
madx.options.warn = False

In [ ]:
madx.input('''SET, format="22.14e";''')

In [ ]:
madx.input('''
qf: multipole, knl={0, 0.3};

testing: sequence, l = 1;
qf, at = 0.5;
endsequence;
''')

In [ ]:
madx.command.beam(
    particle='ion', mass=A*nmass, charge=Q, energy=Etot)

In [ ]:
madx.use(sequence='testing')

In [ ]:
twiss_no_error = madx.twiss(betx=1, bety=1, dx=0, dy=0);

In [ ]:
q1_no_error = twiss_no_error.summary.q1

In [ ]:
madx.select(flag='error', pattern='qf')

In [ ]:
madx.command.efcomp(
    dkn=[0, 1e-2, 5.006767857e-06, 0.01309999342, -0.1972873115, -73.9750876, -9.804122175],
    dks=[0.0, -1.315727925e-05, 1.819912072e-05, -0.002619217458, 0.1995854548, 21.55489111, -919.2177798],
)

In [ ]:
madx.command.ealign(dx=1e-3)

In [ ]:
# qf = madx.sequence.testing.elements[1] 
### --> only "elements" does not work!! need "expanded_elements"
qf = madx.sequence.testing.expanded_elements[2]
qf

In [ ]:
qf.field_errors.dkn, qf.field_errors.dks

In [ ]:
qf.align_errors.dx

In [ ]:
twiss_with_error = madx.twiss(betx=1, bety=1, dx=0, dy=0);

In [ ]:
q1_with_error = twiss_with_error.summary.q1

$\implies$ **The tune has changed with the additional quadrupole error:**

In [ ]:
1 - q1_no_error / q1_with_error

## MAD-X tracking

In [ ]:
madx.input('''
TRACK, onepass, onetable, file=output.;

START, x=1e-03, px=1e-06, y=-1e-03, py=-5e-07, t=0.1, pt=3e-03;
!START, x=0e-03, px=0e-06, y=0e-03, py=0e-07, t=0.1, pt=3e-03;

RUN, turns=1;

ENDTRACK;
''')

In [ ]:
headers = list(np.genfromtxt(
    "output.one", skip_header=51, max_rows=1, dtype=str)[1:])

initial_distribution_madx = np.genfromtxt(
    "output.one", skip_header=54, max_rows=1, dtype=np.float64)

final_distribution_madx = np.genfromtxt(
    "output.one", skip_header=55, max_rows=1, dtype=np.float64)

In [ ]:
def get_betai(PT, mass=mass, p0=p0):
    restmass = mass * c**2
    restmass_sq = restmass**2
    E0 = np.sqrt((p0 * c)**2 + restmass_sq)
    
    E = E0 + PT * p0 * c
    gammai = E / restmass
    betai = np.sqrt(1 - 1. / (gammai * gammai))
    return betai

In [ ]:
pt_madx = final_distribution_madx[headers.index('PT')]
betai = get_betai(pt_madx)

x_madx = final_distribution_madx[headers.index('X')]
px_madx = final_distribution_madx[headers.index('PX')]
y_madx = final_distribution_madx[headers.index('Y')]
py_madx = final_distribution_madx[headers.index('PY')]
zeta_madx = final_distribution_madx[headers.index('T')] * betai
delta_madx = np.sqrt(1 + 2 * pt_madx / beta + pt_madx**2) - 1

## SixTrackLib setup

In [ ]:
madx.esave(file='errors')
madx.command.readtable(file='errors', table='errors')
errors = madx.table.errors

In [ ]:
# PySixTrack, lattice transfer and preparation!

pysixtrack_elements = pysixtrack.Line.from_madx_sequence(
    madx.sequence.testing, exact_drift=True,
)

pysixtrack_elements.apply_madx_errors(errors)

pysixtrack_elements.remove_zero_length_drifts(inplace=True);
pysixtrack_elements.merge_consecutive_drifts(inplace=True);

The PySixTrack multipole has a quadrupole component consisting of the sum of the MAD-X original quadrupole component (0.3) + the error (0.01):

In [ ]:
pysixtrack_elements.elements[2].knl

In [ ]:
pysixtrack_elements.elements[1].dx

In [ ]:
elements = stl.Elements.from_line(pysixtrack_elements)
elements.BeamMonitor(num_stores=1);

In [ ]:
pysixtrack_particles = pysixtrack.Particles.from_madx_track(madx)

In [ ]:
# necessary to transfer to SixTrackLib with this version of PySixTrack
pysixtrack_particles.state = np.array([1, 0])
pysixtrack_particles.elemid = 0

pysixtrack_particles.remove_lost_particles()

In [ ]:
particles = stl.ParticlesSet().Particles(num_particles=1)

particles.from_pysixtrack(pysixtrack_particles, particle_index=0)

In [ ]:
trackjob = stl.TrackJob(elements, particles) #, device="opencl:0.0")

## SixTrackLib tracking

In [ ]:
trackjob.track_until(1)
trackjob.collect()

In [ ]:
x = trackjob.output.particles[0].x[0]
px = trackjob.output.particles[0].px[0]
y = trackjob.output.particles[0].y[0]
py = trackjob.output.particles[0].py[0]
zeta = trackjob.output.particles[0].zeta[0]
delta = trackjob.output.particles[0].delta[0]

# evaluation SixTrackLib vs. MAD-X:

In [ ]:
print ('x error in %: {}'.format(100 * (x - x_madx) / x))

In [ ]:
print ('px error in %: {}'.format(100 * (px - px_madx) / px))

In [ ]:
print ('y error in %: {}'.format(100 * (y - y_madx) / y))

In [ ]:
print ('py error in %: {}'.format(100 * (py - py_madx) / py))

In [ ]:
print ('zeta error in %: {}'.format(100 * (zeta - zeta_madx) / zeta))

In [ ]:
print ('delta error in %: {}'.format(100 * (delta - delta_madx) / delta))

In [ ]:
delta == delta_madx